In [ ]:
import pandas as pd
import os
import sys
sys.path.append('../src')

from preprocessing import get_dfs, create_static_df, create_medication_df

In [ ]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)

In [ ]:
medication = create_medication_df(dfs)
medication

In [ ]:
print(dfs['medikation'].columns)
dfs['medikation'].count()

In [ ]:
print(len(dfs['medikation']))
dfs['medikation']['Bezeichnung'].value_counts().head(10)

In [ ]:
medication = dfs['medikation'][['PatientID', 'TransplantationID', 'prescription start', 'prescription end', 'Bezeichnung', 'DDD', 'unit', 'ATC']].rename(
        columns={
            'PatientID': 'patient_id',
            'TransplantationID': 'transplant_id',
            'prescription start': 'p_start',
            'prescription end': 'p_end',
            'Bezeichnung': 'description',
            'DDD': 'ddd', # defined daily dose
            'unit': 'unit',
            'ATC': 'atc' # the ATC column identifies the medication based on its pharmacological classification.
        }
    )

print(f"Unique patients in medication: {medication['patient_id'].nunique()}")
print('Removing patients that are not in static_df')
medication = medication.merge(
    static_df[['patient_id', 'transplant_id', 'transplant_date']],
    how='inner',  # Inner join to keep only matching entries
    on=['patient_id', 'transplant_id']
)

# Calculate the relative days from transplantation date to prescription start and end
medication['start'] = (pd.to_datetime(medication['p_start'], dayfirst=True)-
                                             pd.to_datetime(medication['transplant_date'], dayfirst=True)
                                              ).dt.days
medication['end'] = (pd.to_datetime(medication['p_end'], dayfirst=True)-
                                           pd.to_datetime(medication['transplant_date'], dayfirst=True)
                                            ).dt.days
medication = medication.drop(columns=['p_start', 'p_end', 'transplant_date'])
print(f"Unique patients in clinical assessments: {medication['patient_id'].nunique()}")
print(f"Average entries per patient {len(medication) / medication['patient_id'].nunique():.1f}")

In [ ]:
medication